# The Multi-Scale Nature of the Solar Wind — Implementation / 태양풍의 다중 스케일 — 구현

**Paper**: Verscharen, Klein, Maruca (2019), *Living Reviews in Solar Physics* 16:5.
**DOI**: 10.1007/s41116-019-0021-0

This notebook reproduces key conceptual results from the review:

1. Solar-wind turbulent power spectrum with MHD (-5/3) + kinetic (-7/3) break at the ion Larmor radius.
2. Ion Larmor radius as a function of plasma β.
3. Maxwell-Boltzmann and anisotropic (bi-Maxwellian) distributions in $v_\|$-$v_\perp$ space.
4. Firehose / mirror instability thresholds in the $(\beta_\|, T_\perp/T_\|)$ plane.
5. Alfvén-wave polarization visualization.

이 노트북은 리뷰 논문의 핵심 개념적 결과를 재현합니다: 난류 파워 스펙트럼, 자이로 반지름 vs β, 분포 함수, 불안정성 문턱, Alfvén 파 편광.

## 0. Setup / 환경 설정

Import libraries and set plotting defaults. Libraries: NumPy for arrays, Matplotlib for figures, SciPy for constants.

라이브러리 임포트와 플로팅 기본 설정.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# Physical constants (SI).
K_B = constants.k          # Boltzmann constant J/K
M_P = constants.m_p        # proton mass kg
M_E = constants.m_e        # electron mass kg
E_CHARGE = constants.e     # elementary charge C
MU0 = constants.mu_0       # permeability H/m

print(f"k_B = {K_B:.3e} J/K")
print(f"m_p = {M_P:.3e} kg, m_e = {M_E:.3e} kg")

## 1. Solar-Wind Turbulent Power Spectrum / 태양풍 난류 파워 스펙트럼

From Eq. (180) of the review, the inertial-range magnetic power spectrum follows Kolmogorov's $k^{-5/3}$. Below the ion Larmor radius $\rho_p$ (spectral break at $k_\perp\rho_p \sim 1$), the spectrum steepens to approximately $k^{-7/3}$ in the kinetic-Alfvén-wave cascade regime (Howes et al. 2008; Alexandrova et al. 2009 give $k^{-2.8}$).

리뷰의 식 (180)에 의해 관성 영역의 자기장 스펙트럼은 $k^{-5/3}$. 이온 자이로 반지름 스케일($k_\perp\rho_p \sim 1$)에서 break가 발생하고 KAW 캐스케이드 영역에서 $k^{-7/3}$로 steepen됨.

**Typical 1 au parameters** (Verscharen et al. 2019, Table 1): $n_p = 5\ \mathrm{cm}^{-3}$, $T_p = 10^5$ K, $B_0 = 5$ nT → $\rho_p \sim 100$ km and spectral break at $f \sim 0.3$–1 Hz via Taylor's hypothesis.

**1 au 전형 파라미터**: $n_p = 5\,\mathrm{cm^{-3}}$, $T_p = 10^5$ K, $B_0 = 5$ nT → $\rho_p \sim 100$ km, Taylor 가설로 spectral break가 $f \sim 0.3$-1 Hz에서 발생.

In [ ]:
def compute_solar_wind_scales(n_p_cm3=5.0, T_p_K=1e5, B_nT=5.0):
    """Compute key solar-wind kinetic scales at 1 au.

    Args:
        n_p_cm3: Proton number density in cm^-3.
        T_p_K: Proton temperature in K.
        B_nT: Magnetic-field magnitude in nT.

    Returns:
        Dictionary with proton thermal speed (m/s), gyrofrequency (rad/s),
        Larmor radius (m), Alfven speed (m/s), and plasma beta.
    """
    n_p = n_p_cm3 * 1e6         # m^-3
    B = B_nT * 1e-9             # Tesla
    w_perp = np.sqrt(2 * K_B * T_p_K / M_P)
    Omega_p = E_CHARGE * B / M_P
    rho_p = w_perp / Omega_p
    v_A = B / np.sqrt(MU0 * n_p * M_P)
    beta_p = 2 * MU0 * n_p * K_B * T_p_K / B ** 2
    return {
        "w_perp": w_perp,
        "Omega_p": Omega_p,
        "rho_p": rho_p,
        "v_A": v_A,
        "beta_p": beta_p,
    }


scales = compute_solar_wind_scales()
print(f"Proton thermal speed w_perp = {scales['w_perp']/1e3:.1f} km/s")
print(f"Proton gyrofrequency Omega_p = {scales['Omega_p']:.3f} rad/s")
print(f"Proton Larmor radius rho_p = {scales['rho_p']/1e3:.1f} km")
print(f"Alfven speed v_A = {scales['v_A']/1e3:.1f} km/s")
print(f"Plasma beta_p = {scales['beta_p']:.3f}")

In [ ]:
def turbulent_spectrum(k_rho_p, amplitude=1.0, mhd_slope=-5/3, kinetic_slope=-7/3):
    """Broken power-law turbulent power spectrum.

    Models the solar-wind magnetic-field power spectrum with a spectral
    break at k_perp * rho_p = 1, transitioning from the MHD cascade
    (Kolmogorov -5/3) to the kinetic cascade (approximately -7/3).

    Args:
        k_rho_p: Dimensionless wavenumber k_perp * rho_p.
        amplitude: Spectrum normalization at k_rho_p = 1.
        mhd_slope: Power-law index in the MHD (inertial) range.
        kinetic_slope: Power-law index in the kinetic range.

    Returns:
        Power spectral density in arbitrary units.
    """
    power = np.where(
        k_rho_p <= 1.0,
        amplitude * k_rho_p ** mhd_slope,
        amplitude * k_rho_p ** kinetic_slope,
    )
    return power


k_rho = np.logspace(-3, 3, 600)
P = turbulent_spectrum(k_rho)

fig, ax = plt.subplots(figsize=(8, 6))
ax.loglog(k_rho, P, 'b-', lw=2, label='Solar wind PSD / 태양풍 PSD')
ax.loglog(k_rho[k_rho < 1], (k_rho[k_rho < 1]) ** (-5 / 3) * 3, 'k--', alpha=0.6,
          label=r'$k^{-5/3}$ (Kolmogorov, MHD)')
ax.loglog(k_rho[k_rho > 1], (k_rho[k_rho > 1]) ** (-7 / 3) * 0.3, 'r--', alpha=0.6,
          label=r'$k^{-7/3}$ (kinetic KAW)')
ax.axvline(1.0, color='gray', ls=':', label=r'$k_\perp\rho_p = 1$ break')
ax.set_xlabel(r'$k_\perp \rho_p$')
ax.set_ylabel('Power spectral density / 파워 스펙트럼 밀도 (arb.)')
ax.set_title('Solar-wind turbulent spectrum / 태양풍 난류 스펙트럼')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 2. Ion Larmor Radius vs Plasma Beta / 이온 자이로 반지름 vs 플라즈마 β

The Larmor radius $\rho_p = w_{\perp p}/\Omega_p$ depends on plasma β via the relation $\rho_p = d_p\sqrt{\beta_p}$, where $d_p = c/\omega_{pp}$ is the proton inertial length (Eq. 8 of review). Hence scanning $\beta_p$ at fixed density traces how the kinetic-scale break location in the turbulent spectrum moves.

자이로 반지름은 β에 따라 $\rho_p = d_p\sqrt{\beta_p}$로 변함. 이 관계에 의해 β가 클수록 kinetic break가 낮은 k로 이동.

In [ ]:
def larmor_vs_beta(beta_range, n_p_cm3=5.0, B_nT=5.0):
    """Compute proton Larmor radius as function of plasma beta.

    Uses the identity rho_p = d_p * sqrt(beta_p), where d_p is the proton
    inertial length determined by n_p and B.

    Args:
        beta_range: Array of plasma beta values to evaluate.
        n_p_cm3: Proton number density in cm^-3 (sets d_p).
        B_nT: Magnetic-field magnitude in nT.

    Returns:
        Tuple (d_p in meters, rho_p in meters across beta_range).
    """
    n_p = n_p_cm3 * 1e6
    eps0 = constants.epsilon_0
    c = constants.c
    omega_pp = np.sqrt(n_p * E_CHARGE ** 2 / (eps0 * M_P))
    d_p = c / omega_pp
    rho_p = d_p * np.sqrt(beta_range)
    return d_p, rho_p


betas = np.logspace(-2, 2, 100)
d_p, rho_p = larmor_vs_beta(betas)

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(betas, rho_p / 1e3, 'b-', lw=2, label=r'$\rho_p = d_p\sqrt{\beta_p}$')
ax.axhline(d_p / 1e3, color='k', ls='--', alpha=0.6, label=r'$d_p$ (inertial length)')
ax.axvline(1, color='gray', ls=':')
ax.set_xlabel(r'$\beta_p$')
ax.set_ylabel(r'$\rho_p$ (km)')
ax.set_title(r'Proton Larmor radius vs plasma $\beta$ / 양성자 자이로 반지름 vs $\beta$')
ax.legend()
plt.tight_layout()
plt.show()

print(f"At beta_p = 1: rho_p = {d_p/1e3:.1f} km (equals d_p)")
print(f"At beta_p = 10: rho_p = {d_p*np.sqrt(10)/1e3:.1f} km")

## 3. Maxwellian and Anisotropic Distributions / Maxwellian 및 비등방 분포

Eq. (59) Maxwellian: $f_M(\mathbf{v}) = n_j/(\pi^{3/2}w_j^3)\exp[-(\mathbf{v}-\mathbf{U}_j)^2/w_j^2]$.

Eq. (61) bi-Maxwellian: $f_{bM}(\mathbf{v}) = n_j/(\pi^{3/2}w_{\perp j}^2 w_{\|j})\exp[-v_\perp^2/w_{\perp j}^2 - (v_\|-U_{\|j})^2/w_{\|j}^2]$.

We contour these in the $v_\|$-$v_\perp$ plane for three cases: isotropic, $T_\perp > T_\|$ (mirror unstable if large enough), and $T_\perp < T_\|$ (firehose unstable).

비등방 분포를 세 경우 (등방, $T_\perp > T_\|$, $T_\perp < T_\|$)로 도시.

In [ ]:
def bi_maxwellian(v_par, v_perp, T_par_ratio=1.0, T_perp_ratio=1.0):
    """Compute bi-Maxwellian distribution value in v_par-v_perp plane.

    Args:
        v_par: Parallel velocity grid normalized by thermal speed.
        v_perp: Perpendicular velocity grid normalized by thermal speed.
        T_par_ratio: T_par / T_baseline (wider -> larger parallel spread).
        T_perp_ratio: T_perp / T_baseline.

    Returns:
        Normalized distribution value f(v_par, v_perp).
    """
    w_par_sq = T_par_ratio
    w_perp_sq = T_perp_ratio
    norm = 1.0 / (np.pi ** 1.5 * w_perp_sq * np.sqrt(w_par_sq))
    return norm * np.exp(-v_par ** 2 / w_par_sq - v_perp ** 2 / w_perp_sq)


v = np.linspace(-3, 3, 200)
V_par, V_perp = np.meshgrid(v, v)

cases = [
    (r"Isotropic / 등방 ($T_\perp/T_\|=1$)", 1.0, 1.0),
    (r"Mirror-unstable / 거울 불안정 ($T_\perp/T_\|=3$)", 1.0, 3.0),
    (r"Firehose-unstable / firehose 불안정 ($T_\perp/T_\|=1/3$)", 3.0, 1.0),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, T_par, T_perp) in zip(axes, cases):
    f = bi_maxwellian(V_par, V_perp, T_par, T_perp)
    cs = ax.contourf(V_par, V_perp, f, levels=20, cmap='viridis')
    ax.contour(V_par, V_perp, f, levels=6, colors='k', linewidths=0.6, alpha=0.7)
    ax.set_xlabel(r'$v_\| / w_0$')
    ax.set_ylabel(r'$v_\perp / w_0$')
    ax.set_title(title)
    ax.set_aspect('equal')
    plt.colorbar(cs, ax=ax, label=r'$f(v_\|, v_\perp)$')
plt.tight_layout()
plt.show()

## 4. Firehose / Mirror Instability Thresholds / Firehose·Mirror 불안정성 문턱

Using Eq. (198) of the review with Hellinger et al. (2006) coefficients (Table 3) at max growth rate $\gamma_m = 10^{-2}\Omega_p$:

$$\frac{T_\perp}{T_\|} = 1 + \frac{a}{(\beta_\| - c)^b}$$

| Instability | a | b | c |
|---|---|---|---|
| Ion-cyclotron | 0.649 | 0.400 | 0.000 |
| Mirror | 1.040 | 0.633 | -0.012 |
| Parallel firehose | -0.647 | 0.583 | 0.713 |
| Oblique firehose | -1.447 | 1.000 | -0.148 |

This reproduces the 'Brazil plot' boundaries of Fig. 21 of Verscharen et al. (2019).

리뷰 식 (198)과 Hellinger et al. (2006) 계수로 Brazil plot 경계 재현.

In [ ]:
def anisotropy_threshold(beta_par, a, b, c):
    """Compute instability threshold T_perp/T_par.

    Parametric fit from Hellinger et al. (2006), Eq. (198) of review.

    Args:
        beta_par: Parallel plasma beta array.
        a, b, c: Fit parameters (Table 3 of review).

    Returns:
        Threshold T_perp / T_par as a function of beta_par.
    """
    base = beta_par - c
    safe = np.where(base > 1e-6, base, np.nan)
    return 1 + a / (safe ** b)


thresholds = {
    'Ion-cyclotron / 이온-사이클로트론': (0.649, 0.400, 0.000),
    'Mirror / 거울': (1.040, 0.633, -0.012),
    'Parallel firehose / 평행 firehose': (-0.647, 0.583, 0.713),
    'Oblique firehose / 비스듬 firehose': (-1.447, 1.000, -0.148),
}

beta_range = np.logspace(-2, 1.3, 400)
fig, ax = plt.subplots(figsize=(9, 7))
colors = ['blue', 'red', 'green', 'purple']

for (name, (a, b, c)), color in zip(thresholds.items(), colors):
    threshold = anisotropy_threshold(beta_range, a, b, c)
    ax.plot(beta_range, threshold, color=color, lw=2, label=name)

ax.axhline(1.0, color='gray', ls=':', alpha=0.6)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(1e-2, 20)
ax.set_ylim(0.1, 10)
ax.set_xlabel(r'$\beta_{\|p}$')
ax.set_ylabel(r'$T_{\perp p} / T_{\|p}$')
ax.set_title(r'Proton anisotropy stability thresholds / 양성자 비등방성 안정 문턱')
ax.text(0.03, 3, 'Mirror\nunstable', fontsize=9, color='red')
ax.text(0.03, 0.3, 'Parallel firehose', fontsize=9, color='green')
ax.text(5, 0.2, 'Oblique firehose', fontsize=9, color='purple')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

## 5. Alfvén-Wave Polarization / Alfvén 파 편광

For a large-scale Alfvén wave (Eq. 163), the velocity and magnetic-field fluctuations satisfy:
$$\delta\mathbf{U}/v_A^* = \mp\delta\mathbf{B}/B_0$$

where the upper/lower sign corresponds to parallel/anti-parallel propagation relative to $\mathbf{B}_0$. We visualize the in-phase correlation $\delta U_y = -v_A\delta B_y/B_0$ for a parallel-propagating wave with amplitude 30% and $v_A = 50$ km/s.

Alfvén 파의 $\delta\mathbf{U}$-$\delta\mathbf{B}$ 반상관 관계를 시간 시리즈로 시각화.

In [ ]:
def alfven_wave(t, amplitude=0.3, v_A_km_s=50.0, B0_nT=5.0, omega=2 * np.pi / 60):
    """Generate synthetic Alfven-wave time series in velocity and magnetic field.

    Implements the MHD Alfven-wave polarization relation
    delta U / v_A = - delta B / B0 for a parallel-propagating wave.

    Args:
        t: Time array in seconds.
        amplitude: Relative magnetic amplitude delta B / B0 (fraction).
        v_A_km_s: Alfven speed in km/s.
        B0_nT: Background magnetic-field magnitude in nT.
        omega: Angular wave frequency in rad/s.

    Returns:
        Tuple of (delta U in km/s, delta B in nT).
    """
    dB = amplitude * B0_nT * np.sin(omega * t)
    dU = -v_A_km_s * dB / B0_nT
    return dU, dB


t = np.linspace(0, 300, 3000)
dU, dB = alfven_wave(t, amplitude=0.3)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
ax1.plot(t, dU, 'b-', lw=1.5, label=r'$\delta U_y$ (km/s)')
ax1.set_ylabel(r'$\delta U_y$ (km/s)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1b = ax1.twinx()
ax1b.plot(t, dB, 'r-', lw=1.5, alpha=0.7, label=r'$\delta B_y$ (nT)')
ax1b.set_ylabel(r'$\delta B_y$ (nT)', color='red')
ax1b.tick_params(axis='y', labelcolor='red')
ax1.set_xlabel('Time (s) / 시간')
ax1.set_title(r'Alfvenic correlation $\delta U = -v_A\delta B/B_0$ / 알펜 반상관')
ax1.grid(alpha=0.3)

ax2.scatter(dB, dU, s=5, c=t, cmap='viridis')
ax2.set_xlabel(r'$\delta B_y$ (nT)')
ax2.set_ylabel(r'$\delta U_y$ (km/s)')
ax2.set_title(r'Hodograph: perfect anti-correlation expected / 호도그램: 완벽한 반상관')
ax2.grid(alpha=0.3)
v_A = 50.0
B0 = 5.0
ax2.plot(dB, -v_A * dB / B0, 'k--', alpha=0.6, label=r'$\delta U = -(v_A/B_0)\delta B$')
ax2.legend()
plt.tight_layout()
plt.show()

## Summary / 요약

We have implemented five key conceptual results from Verscharen, Klein, Maruca (2019):

1. **Turbulent spectrum** with MHD $k^{-5/3}$ and kinetic $k^{-7/3}$ separated at $k_\perp\rho_p = 1$ — the hallmark of the MHD-to-kinetic cascade transition.
2. **Larmor radius $\rho_p = d_p\sqrt{\beta_p}$** showing how the kinetic-scale break shifts with plasma β.
3. **Bi-Maxwellian distributions** in $v_\|$–$v_\perp$ for isotropic / mirror-unstable / firehose-unstable cases.
4. **Hellinger-Bale Brazil-plot thresholds** (ion-cyclotron, mirror, parallel/oblique firehose) using Table 3 coefficients.
5. **Alfvén-wave polarization** demonstrating the $\delta\mathbf{U}$–$\delta\mathbf{B}$ anti-correlation.

These elements together illustrate the multi-scale framework: the cascade hands energy from MHD scales down to $\rho_p$, where kinetic physics (instabilities, wave–particle resonances) takes over and feeds energy back into particle populations.

이 구현은 리뷰 논문의 다중 스케일 틀을 핵심 5가지 개념으로 요약: (1) 난류 스펙트럼의 MHD→kinetic 전환, (2) β에 따른 자이로 반지름 변화, (3) 비등방 분포, (4) 불안정성 문턱, (5) Alfvén 파 편광.